In [27]:
!pip install clustering-benchmarks -q

In [28]:
import clustbench
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

In [29]:
battery_datasets_dict = {
                         'fcps': ['atom', 'chainlink', 'engytime', 'hepta', 'lsun', 'target', 'tetra', 'twodiamonds', 'wingnut'],
                         'uci': ['ecoli', 'glass', 'ionosphere', 'sonar', 'statlog', 'wdbc', 'wine', 'yeast'],
                         #'mnist': ['digits', 'fashion'], # Genie digits ~ 32 min
                         #'sipu': ['worms_64'], # Genie ~ 9 min
                         }

| Battery | Dataset    | Rows   | Cols   | Clusters
|---------|------------|--------|--------|----------
| uci     | ecoli      | 336    | 7      | 8
| uci     | glass      | 214    | 9      | 6
| uci     | ionosphere | 351    | 33     | 2
| uci     | sonar      | 208    | 60     | 2
| uci     | statlog    | 2310   | 18     | 7
| uci     | wdbc       | 569    | 30     | 2
| uci     | wine       | 178    | 13     | 3
| uci     | yeast      | 1484   | 8      | 4
| mnist   | digits     | 70000  | 719    | 10
| mnist   | fashion    | 70000  | 784    | 10
| sipu    | worms_64   | 105000 | 64     | 25


In [30]:
import numpy as np
from sklearn.neural_network import MLPRegressor
import pandas as pd

def ssnn_feat_eng(X, n_embeddings=2):

  # To store embeddings for each feature
  embeddings_list = []

  # Get the number of features
  n_samples, n_features = X.shape

  for feature_idx in range(n_features):

      # Prepare input by removing the current feature
      X_in = np.delete(X, feature_idx, axis=1)  # shape: (n_samples, n_features-1)

      # Target is the left-out feature
      y_target = X[:, feature_idx]  # shape: (n_samples,)

      # Define MLP
      mlp = MLPRegressor(
          hidden_layer_sizes=(64, n_embeddings),
          # Use 'identity' for linear activation in the hidden layers
          activation='logistic', #{'relu', 'logistic', 'identity', 'tanh'}
          max_iter=10000,
          random_state=42
      )

      # Fit the model
      mlp.fit(X_in, y_target)

      # Extract learned weights and biases
      coefs = mlp.coefs_
      intercepts = mlp.intercepts_

      # Forward pass to hidden layer 1 (linear)
      Z1 = X_in @ coefs[0] + intercepts[0]  # shape: (n_samples, 64)

      # Forward pass to hidden layer 2 (linear) → this is the embedding
      embeddings = Z1 @ coefs[1] + intercepts[1]  # shape: (n_samples, n_embeddings)

      # Append embeddings
      embeddings_list.append(embeddings)

  # Concatenate all embeddings horizontally
  X_new = np.hstack(embeddings_list)  # shape: (n_samples, n_embeddings * n_features)
  #X_new_df = pd.DataFrame(X_new, columns=[f'feature_{i+1}_emb_{j+1}' for i in range(n_features) for j in range(n_embeddings)])
  return X_new

In [31]:
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"
battery = "uci"
dataset = "statlog"
b = clustbench.load_dataset(battery, dataset, url=data_url)

In [32]:
pd.DataFrame(b.data)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,1.765322,1.035128,0.001835,-0.000089,-0.020114,-0.097886,-0.024911,-0.146014,0.428179,0.372142,0.588547,0.323848,-0.168113,0.481107,-0.312992,0.570539,-0.002046,-0.012852
1,-0.225939,0.124837,-0.000272,-0.000089,-0.030649,-0.103516,-0.039661,-0.149411,-0.685803,-0.622435,-0.789533,-0.645442,0.190103,-0.311185,0.121082,-0.807539,0.010869,-0.014420
2,1.461891,-1.562993,-0.000273,-0.000090,-0.018007,-0.093629,-0.024913,-0.136886,1.630661,1.499467,1.812804,1.579711,-0.393578,0.546427,-0.152849,1.794795,-0.004316,-0.017771
3,-1.762054,0.940306,-0.000273,-0.000089,-0.003256,-0.074487,0.124697,-0.028337,0.124045,0.127711,0.165010,0.079419,0.010994,0.122890,-0.133885,0.147000,-0.003034,-0.012060
4,-1.212087,1.395450,-0.000272,-0.000090,-0.008523,-0.079536,0.003536,-0.119820,0.237831,0.216211,0.329367,0.167919,-0.064863,0.274605,-0.209741,0.311358,-0.002350,-0.012504
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2305,-1.799983,-0.406166,-0.000271,-0.000089,-0.012739,-0.106025,-0.020698,-0.141165,-0.318457,-0.236827,-0.363887,-0.354655,0.244889,-0.136290,-0.108598,-0.381895,-0.000869,-0.003645
2306,0.342994,-1.885389,-0.000272,-0.000089,-0.011685,-0.091066,-0.029126,-0.134703,1.717757,1.609040,1.848625,1.695606,-0.326149,0.392604,-0.066456,1.830616,-0.004883,-0.018706
2307,-0.851765,-0.975098,-0.000272,-0.000088,-0.012740,-0.089239,-0.018589,-0.134195,0.416238,0.351069,0.573796,0.323848,-0.195507,0.472677,-0.277171,0.555789,-0.002128,-0.013794
2308,-0.510405,0.181730,-0.000272,-0.000090,-0.025381,-0.105009,-0.038608,-0.150122,-0.684399,-0.622435,-0.785316,-0.645442,0.185888,-0.302756,0.116866,-0.803325,0.010870,-0.014420


In [33]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores(battery, dataset, apply_scale=False):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

In [34]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_ssnn(battery, dataset, apply_scale=False, n_embeddings=2):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = ssnn_feat_eng(b.data, n_embeddings=n_embeddings)

  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

In [35]:
import tqdm
import pandas as pd
columns = ['Battery', 'Dataset', 'Genie NCA Score']
df = pd.DataFrame(columns=columns)
scores_lists = {}
for col in columns:
  scores_lists[col] = []

for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Datasets"):
  for dataset in battery_datasets_dict[battery]:
    scores_lists['Battery'].append(battery)
    scores_lists['Dataset'].append(dataset)
    scores_lists['Genie NCA Score'].append(get_scores(battery, dataset))

df = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 2/2 [00:04<00:00,  2.33s/it]


In [36]:
df

,Battery,Dataset,Genie NCA Score
0,fcps,atom,1.000000
1,fcps,chainlink,1.000000
2,fcps,engytime,0.918870
3,fcps,hepta,1.000000
4,fcps,lsun,1.000000
5,fcps,target,1.000000
6,fcps,tetra,1.000000
7,fcps,twodiamonds,0.987500
8,fcps,wingnut,1.000000
9,uci,ecoli,0.435664


In [37]:
n_embeddings_list = [2,4,8,16,32]

scores_lists = {}
for n_embs in tqdm.tqdm(n_embeddings_list, desc="Processing Datasets"):
  for battery in battery_datasets_dict.keys():
    for dataset in battery_datasets_dict[battery]:
      column_name = 'Genie+SSNN '+ str(n_embs) +' NCA Score'
      if column_name not in scores_lists:
        scores_lists[column_name] = []
      scores_lists[column_name].append(get_scores_with_ssnn(battery, dataset, n_embeddings=n_embs))

df_ssnn = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 5/5 [08:21<00:00, 100.28s/it]


In [39]:
df = pd.concat([df, df_ssnn], axis=1)
numerical_cols = df.columns[2:]
df.style.highlight_max(axis=1, subset=numerical_cols)

,Battery,Dataset,Genie NCA Score,Genie+SSNN 2 NCA Score,Genie+SSNN 4 NCA Score,Genie+SSNN 8 NCA Score,Genie+SSNN 16 NCA Score,Genie+SSNN 32 NCA Score
0,fcps,atom,1.000000,0.977500,1.000000,1.000000,1.000000,1.000000
1,fcps,chainlink,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,fcps,engytime,0.918870,0.918870,0.942347,0.918870,0.694877,0.690478
3,fcps,hepta,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,fcps,lsun,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
5,fcps,target,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,fcps,tetra,1.000000,0.736667,0.560000,0.993333,1.000000,1.000000
7,fcps,twodiamonds,0.987500,0.987500,0.987500,0.987500,0.992500,0.992500
8,fcps,wingnut,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,uci,ecoli,0.435664,0.405951,0.392308,0.375125,0.300200,0.376716
